In [1]:
!pip install -U \
langchain \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain-google-genai \
langchain-huggingface \
sentence-transformers \
chromadb \
pandas -q \
langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.2/111.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.3/496.3 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Standard libraries
import os
import json
import pandas as pd
from typing import List
import re


from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.chat_message_histories import ChatMessageHistory

# LangChain core
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder



# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector store
from langchain_community.vectorstores import Chroma

# Memory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# LLMs & Embeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings


In [ ]:
# Set your Google Gemini API Key
os.environ["GOOGLE_API_KEY"] = ""



In [4]:
# Load and prepare colleges / departments data
# Optimized for semantic search (Arabic main + English keywords)

college_docs = []

with open("/content/chatbot_final_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for idx, item in enumerate(data):
    meta = item.get("metadata", {})

    # Arabic (main understanding language)
    university_ar = meta.get("university", {}).get("ar", "")
    faculty_ar = meta.get("faculty", {}).get("ar", "")
    department_ar = meta.get("department", {}).get("ar", "")
    description_ar = item.get("page_content", {}).get("ar", "")

    interests_ar = ", ".join(meta.get("interests", {}).get("ar", []))
    subjects_ar = ", ".join(meta.get("strong_subjects", {}).get("ar", []))
    skills_ar = ", ".join(meta.get("required_skills", {}).get("ar", []))
    careers_ar = ", ".join(meta.get("career_paths", {}).get("ar", []))
    learning_style_ar = ", ".join(meta.get("learning_style", {}).get("ar", []))

    # English (keywords only – for better embedding recall)
    department_en = meta.get("department", {}).get("en", "")
    interests_en = ", ".join(meta.get("interests", {}).get("en", []))
    skills_en = ", ".join(meta.get("required_skills", {}).get("en", []))

    content = f"""
    الجامعة: {university_ar}
    الكلية: {faculty_ar}
    القسم: {department_ar}

    وصف البرنامج:
    {description_ar}

    الاهتمامات:
    {interests_ar}

    المواد الأساسية:
    {subjects_ar}

    المهارات المطلوبة:
    {skills_ar}

    أسلوب الدراسة:
    {learning_style_ar}

    المسارات المهنية:
    {careers_ar}

    --- English Keywords ---
    Department: {department_en}
    Interests: {interests_en}
    Skills: {skills_en}
    """

    college_docs.append(
        Document(
            page_content=content.strip(),
            metadata={
                "id": item.get("id", f"college_{idx}"),
                "type": "faculty_department_profile"
            }
        )
    )



Loaded 483 college / department documents


In [5]:
# Load and prepare career tracks
# Clean version: only what matters for recommendation & search

df = pd.read_excel("/content/all_tracks.xlsx").fillna("")

track_docs = []

for _, row in df.iterrows():

    track_name_ar = row.get("Track_Name_Arabic", "")
    track_name_en = row.get("Track_Name_English", "")
    category_ar = row.get("Category_Arabic", "")
    level = row.get("Level", "")
    duration = row.get("Duration_Months", "")

    core_skills = row.get("Core_Skills", "")
    soft_skills_ar = row.get("Soft_Skills_Arabic", "")
    job_roles = row.get("Job_Roles", "")

    content = f"""
    المسار المهني:
    {track_name_ar}

    الفئة:
    {category_ar}

    المستوى:
    {level}

    مدة التعلم:
    {duration} شهر

    المهارات الأساسية:
    {core_skills}

    المهارات السلوكية:
    {soft_skills_ar}

    الوظائف المحتملة:
    {job_roles}

    --- English Keywords ---
    Track: {track_name_en}
    """

    track_docs.append(
        Document(
            page_content=content.strip(),
            metadata={
                "Track_Name_Arabic": track_name_ar,
                "type": "career_track"
            }
        )
    )


In [6]:
# Initialize embedding model and text splitter

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
college_vs = Chroma.from_documents(
    splitter.split_documents(college_docs),
    embedding=embeddings,
    collection_name="colleges"
)

track_vs = Chroma.from_documents(
    splitter.split_documents(track_docs),
    embedding=embeddings,
    collection_name="tracks"
)

print("Vector stores ready")

Vector stores ready


In [8]:


# from langchain_openai import ChatOpenAI
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

# llm = ChatOpenAI(
#     model="llama-3.1-8b-instant",
#     openai_api_base="https://api.groq.com/openai/v1",
#     openai_api_key=os.environ["GROQ_API_KEY"],
#     temperature=0.7,
# )


In [9]:
# CAREER GUIDANCE SYSTEM - COMPLETE WORKFLOW





# STEP 1: Get User Information

print("Welcome to Career and Academic Guidance System")


user_name = input("Please enter your name: ")

#### هنا نجيب اسم اليوزر من الباااااااااااااك


print(f"\nHello {user_name}! Welcome.")
print()


# STEP 2: Determine User Path (Tracks Only or College + Tracks)

print("Let's determine your needs:")
print()
print("1  I'm looking for a suitable training track for my skills")
print()
print("2  I'm a high school graduate and want to know suitable colleges and majors")

##هنا تكون BUTTON  بدل الاختيار دي

user_path = input("Choose (1 or 2): ").strip()

while user_path not in ['1', '2']:
    print("Please choose 1 or 2 only")
    user_path = input("Choose (1 or 2): ").strip()

print()

if user_path == '1':
    path_type = "track_only"
    print(f"Excellent {user_name}! We will help you choose the right track.")
else:
    path_type = "college_and_track"
    print(f"Great {user_name}! We will help you choose the right college and track.")

print()


# STEP 3: Setup Interview Questions Based on Path Type

total_questions = 4
conversation_history = []

if path_type == "track_only":
    system_prompt = f"""
أنت مستشار مسارات تدريبية محترف ومتخصص في توجيه الأفراد لاختيار التراك التدريبي الأنسب لهم.

اسم الشخص: {user_name}
هدف المقابلة: مساعدة {user_name} في تحديد التراك التدريبي الأكثر توافقًا مع مهاراته واهتماماته وطموحاته العملية.

هيكل المقابلة:
- المقابلة مكونة من {total_questions} أسئلة فقط.
- أنت تجمع معلومات ولا تقدم أي اقتراحات أو توصيات في هذه المرحلة.

الأهداف التي يجب استخلاصها بنهاية المقابلة:
- المهارات الحالية (التقنية أو غير التقنية)
- الاهتمامات العملية والمجالات المفضلة
- مستوى الخبرة (مبتدئ / متوسط / متقدم)
- الوقت المتاح والقدرة على الالتزام
- أسلوب التدريب المفضل (عملي / نظري / مختلط)
- الهدف النهائي من التعلم (وظيفة – تطوير ذات – عمل حر)

قواعد صارمة:
- اسأل سؤالًا واحدًا فقط في كل مرة
- كل سؤال يجب أن يبنى على الإجابات السابقة
- استخدم اسم {user_name} داخل السؤال بشكل طبيعي
- أسلوبك يجب أن يكون Interview حقيقي (واضح – مباشر – ودود)
- ركّز على الجانب التطبيقي والمهني
- لا تقدّم أي ترشيحات أو أسماء تراكات الآن
"""

else:
    system_prompt = f"""
أنت مستشار أكاديمي ومهني محترف متخصص في توجيه طلاب الثانوية لاختيار المسار الجامعي المناسب.

اسم الشخص: {user_name}
هدف المقابلة: مساعدة {user_name} في اختيار الكلية والتخصص الأنسب بناءً على ميوله وقدراته وخططه المستقبلية.

هيكل المقابلة:
- المقابلة مكونة من {total_questions} أسئلة فقط.
- هذه مرحلة فهم وتحليل فقط، بدون تقديم توصيات.

الأهداف التي يجب استخلاصها بنهاية المقابلة:
- الاهتمامات الأكاديمية والمواد المفضلة
- نقاط القوة الدراسية
- المجالات التي يفضل دراستها بعمق
- الأهداف المهنية بعد التخرج
- أسلوب التعلم المفضل (نظري / عملي / بحثي)
- القدرة النفسية والزمنية على الالتزام بدراسة جامعية طويلة

قواعد صارمة:
- اسأل سؤالًا واحدًا فقط في كل مرة
- كل سؤال يجب أن يعتمد على الإجابات السابقة
- استخدم اسم {user_name} لجعل الحوار شخصيًا
- أسلوبك Interview حقيقي (واضح – مباشر – ودود)
- ركّز على الميول الأكاديمية والمهنية معًا
- لا تقدّم أي اقتراحات أو أسماء كليات الآن
"""



# STEP 4: Conduct Interview Loop

print(f"Starting interview with {user_name}")
print()

for current_question in range(1, total_questions + 1):

    # Prepare summary of previous answers
    answers_summary = ""
    for item in conversation_history:
        if item['answer']:
            answers_summary += f"""
السؤال {item['question_number']}: {item['question']}
الإجابة: {item['answer']}
"""

    # Prepare prompt based on question number
    if current_question == 1:
        prompt = f"""
أنت الآن في بداية المقابلة مع {user_name}.

السؤال الحالي: {current_question} من {total_questions}
الأسئلة المتبقية: {total_questions - current_question}

لا توجد إجابات سابقة بعد.

اطرح السؤال الأول لـ {user_name}.
{"يجب أن يكون سؤالاً عن المهارات الحالية والاهتمامات التدريبية." if path_type == "track_only" else "يجب أن يكون سؤالاً عن الاهتمامات الأكاديمية والمواد المفضلة في الثانوية."}
"""
    else:
        last_answer = conversation_history[-1]['answer']

        prompt = f"""
السؤال الحالي: {current_question} من {total_questions}
الأسئلة المتبقية: {total_questions - current_question}

ملخص المقابلة مع {user_name} حتى الآن:
{answers_summary.strip()}

آخر إجابة من {user_name}:
"{last_answer}"

مهمتك الآن:
بناءً على كل ما سبق، اطرح السؤال رقم {current_question} لـ {user_name}.
يجب أن يكون السؤال:
- مبنياً على إجابات {user_name} السابقة
- يغطي جانباً جديداً لم نتطرق له
- واضحاً ومباشراً وودوداً
"""

    # Call LLM
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=prompt)
    ]

    response = llm.invoke(messages)
    question_text = response.content

    # Display question
    print(f"Question {current_question} of {total_questions}:")
    print(question_text)
    print("\n" + "="*60)

    # Get user answer
    user_answer = input(f"{user_name}: ")

    # Save question and answer
    conversation_history.append({
        'question_number': current_question,
        'question': question_text,
        'answer': user_answer
    })

    print(f"\nThank you {user_name}! Answer saved.")
    print()


# STEP 5: Analysis Phase - Extract Keywords

print("Analyzing interview answers...")
print()

# Prepare answers summary for analysis
answers_summary = ""
for item in conversation_history:
    answers_summary += f"""
السؤال {item['question_number']}: {item['question']}
الإجابة: {item['answer']}
───────────────────────────────────
"""







Welcome to Career and Academic Guidance System
Please enter your name: علي

Hello علي! Welcome.

Let's determine your needs:

1  I'm looking for a suitable training track for my skills

2  I'm a high school graduate and want to know suitable colleges and majors
Choose (1 or 2): 2

Great علي! We will help you choose the right college and track.

Starting interview with علي

Question 1 of 4:
أهلاً بك يا علي، سعيد بمقابلتك اليوم. لمساعدتك في التفكير في مستقبلك الجامعي، دعنا نبدأ بالحديث عن دراستك في الثانوية.

ما هي المواد الدراسية التي كنت تستمتع بدراستها أكثر من غيرها في المرحلة الثانوية، وما الذي كان يجعلك تفضلها؟

علي: رياضيات وفيزياء

Thank you علي! Answer saved.

Question 2 of 4:
أهلاً بك يا علي. جميل أنك تستمتع بالرياضيات والفيزياء. لمساعدتي في فهم أعمق لميولك، هل يمكنك أن تخبرني ما الذي يجذبك تحديداً في هذين المادتين؟ هل هو حل المسائل المعقدة، أم فهم النظريات والقوانين، أم رؤية تطبيقاتها العملية؟

علي: نعم حل المسائل المعقدة مع تطبيق القوانين 

Thank you علي! Answer saved.

Questi

ValueError: Invalid format specifier ' [
    "اهتمام تطبيقي واضح",
    "مجال عملي مفضل",
    "نشاط مهني متكرر"
  ],
  "skills": [
    "مهارة تقنية أو مهنية",
    "أداة أو تقنية مذكورة",
    "مهارة عملية مستنتجة من السياق"
  ],
  "career_goals": [
    "هدف مهني قصير المدى",
    "هدف مهني متوسط أو طويل المدى"
  ],
  "search_queries_arabic": [
    "كلمة بحث عربية دقيقة",
    "مصطلح تدريبي مستخدم في السوق",
    "مجال تدريبي واضح"
  ],
  "search_queries_english": [
    "technical keyword",
    "training track keyword",
    "industry-related term"
  ],
  "preferred_tracks": [
    "اسم تراك محتمل (عام بدون افتراضات)",
    "اسم تراك محتمل آخر"
  ]
' for object of type 'str'

In [10]:
# ANALYSIS 1: Extract Keywords for Tracks

print("Analysis 1: Extracting keywords for training tracks...")
print()

track_analysis_prompt = f"""
لديك مقابلة مكتملة مع {user_name} بهدف اختيار مسار تدريبي مناسب (Track Only).

إجابات المقابلة:
{answers_summary.strip()}

مهمتك:
اقرأ جميع الإجابات أولاً، ثم حلّلها كمستشار مهني، وليس كمجرد استخراج كلمات.
حوّل ناتج التحليل إلى ملف بحث منظم يصلح للبحث داخل قاعدة بيانات التراكات التدريبية.

تعليمات التحليل:
- استخرج الاهتمامات العملية والتطبيقية المتكررة فقط.
- استخرج المهارات التقنية أو المهنية المذكورة صراحة أو ضمنيًا.
- استنتج أهدافًا مهنية واقعية بناءً على السياق الكامل.
- حوّل النتائج إلى كلمات مفتاحية دقيقة قابلة للاستخدام في Semantic Search.
- استخدم العربية والإنجليزية كما تُستخدم فعليًا في سوق العمل.
- لا تفترض أي مهارات أو اهتمامات غير مدعومة بالإجابات.

قواعد صارمة:
- أعد النتيجة بصيغة JSON فقط.
- لا تضف أي نص أو شرح خارج JSON.
- لا تستخدم Markdown أو ```json.
- جميع القيم Arrays فقط.
- لا تكرر العناصر داخل نفس القائمة.

صيغة الإخراج المطلوبة:

{{
  "interests": [
    "اهتمام تطبيقي واضح",
    "مجال عملي مفضل",
    "نشاط مهني متكرر"
  ],
  "skills": [
    "مهارة تقنية أو مهنية",
    "أداة أو تقنية مذكورة",
    "مهارة عملية مستنتجة من السياق"
  ],
  "career_goals": [
    "هدف مهني قصير المدى",
    "هدف مهني متوسط أو طويل المدى"
  ],
  "search_queries_arabic": [
    "كلمة بحث عربية دقيقة",
    "مصطلح تدريبي مستخدم في السوق",
    "مجال تدريبي واضح"
  ],
  "search_queries_english": [
    "technical keyword",
    "training track keyword",
    "industry-related term"
  ],
  "preferred_tracks": [
    "اسم تراك محتمل عام بدون افتراضات",
    "اسم تراك محتمل آخر"
  ]
}}
"""

messages = [
    SystemMessage(content="أنت محلل مهني يستخرج JSON خام فقط دون أي نص إضافي."),
    HumanMessage(content=track_analysis_prompt)
]

track_analysis_response = llm.invoke(messages)

track_analysis_data = None
try:
    cleaned = track_analysis_response.content.strip()
    cleaned = re.sub(r'```json\s*|\s*```', '', cleaned)
    track_analysis_data = json.loads(cleaned)

    print("Successfully extracted track keywords:")
    print(json.dumps(track_analysis_data, ensure_ascii=False, indent=2))
except Exception:
    print("Failed to extract JSON for tracks:")
    print(track_analysis_response.content)

print()


# ANALYSIS 2: Extract Keywords for Colleges (Only if path is college_and_track)

college_analysis_data = None
college_analysis_response = None

if path_type == "college_and_track":
    print("Analysis 2: Extracting keywords for colleges and academic departments...")
    print()

    college_analysis_prompt = f"""
لديك مقابلة مكتملة مع {user_name}، خريج مرحلة ثانوية، بهدف اختيار الكلية والقسم الجامعي الأنسب له.

إجابات المقابلة:
{answers_summary.strip()}

مهمتك:
حلّل الإجابات بالكامل كمرشد أكاديمي، ثم حوّلها إلى ملف بحث منظم للبحث داخل قواعد بيانات الكليات والأقسام.

تعليمات التحليل:
- استخرج الاهتمامات الأكاديمية الحقيقية (غير العامة).
- حدّد المواد أو المجالات التي يظهر فيها ميل أو تميز.
- استنتج المجالات العلمية أو البحثية المحتملة.
- استخرج المهارات الدراسية والذهنية المرتبطة بالنجاح الجامعي.
- استخدم مصطلحات عربية وإنجليزية شائعة في توصيف التخصصات الجامعية.
- لا تفترض أي ميول أو قدرات غير مدعومة بالإجابات.

قواعد صارمة:
- JSON فقط دون أي نص إضافي.
- لا Markdown ولا ```json.
- جميع القيم Arrays فقط.
- تجنب التكرار أو التعميم.
- التزم بما ورد في الإجابات فقط.

صيغة الإخراج المطلوبة:

{{
  "interests": [
    "اهتمام أكاديمي واضح",
    "مجال علمي مفضل",
    "تخصص دراسي يميل إليه"
  ],
  "skills": [
    "مهارة دراسية أو تحليلية",
    "قدرة ذهنية مرتبطة بالدراسة الجامعية",
    "مهارة تعلم أو بحث"
  ],
  "career_goals": [
    "هدف مهني بعد التخرج",
    "مسار مهني مستقبلي محتمل"
  ],
  "search_queries_arabic": [
    "اسم كلية أو تخصص بالعربية",
    "مجال أكاديمي مستخدم في الجامعات",
    "مصطلح دراسي شائع"
  ],
  "search_queries_english": [
    "academic major keyword",
    "college department term",
    "field of study keyword"
  ],
  "preferred_departments": [
    "اسم قسم أو كلية محتملة عامة",
    "اسم قسم أو كلية محتملة أخرى"
  ]
}}
"""

    messages = [
        SystemMessage(content="أنت محلل أكاديمي يستخرج JSON خام فقط دون أي نص إضافي."),
        HumanMessage(content=college_analysis_prompt)
    ]

    college_analysis_response = llm.invoke(messages)

    try:
        cleaned = college_analysis_response.content.strip()
        cleaned = re.sub(r'```json\s*|\s*```', '', cleaned)
        college_analysis_data = json.loads(cleaned)

        print("Successfully extracted college keywords:")
        print(json.dumps(college_analysis_data, ensure_ascii=False, indent=2))
    except Exception:
        print("Failed to extract JSON for colleges:")
        print(college_analysis_response.content)

    print()
else:
    print("Skipping college analysis (user selected tracks only)")
    print()

print("Analysis complete")
print()


Analysis 1: Extracting keywords for training tracks...

Successfully extracted track keywords:
{
  "interests": [
    "حل المسائل المعقدة",
    "تطبيق القوانين والنظريات",
    "حل المشكلات الواقعية",
    "تصميم وبناء حلول ملموسة",
    "تسهيل حياة البشر"
  ],
  "skills": [
    "حل المشكلات",
    "التفكير التحليلي",
    "تطبيق المبادئ العلمية",
    "التصميم الهندسي",
    "التفكير المنطقي"
  ],
  "career_goals": [
    "العمل في مجال تطبيقي",
    "تطوير حلول مبتكرة للمشكلات",
    "المساهمة في تحسين حياة الناس",
    "استخدام المهارات التقنية في مجال عملي"
  ],
  "search_queries_arabic": [
    "هندسة تطبيقية",
    "حلول تقنية",
    "تطوير منتجات",
    "تصميم هندسي",
    "تكنولوجيا تطبيقية",
    "هندسة برمجيات",
    "ذكاء اصطناعي تطبيقي",
    "علم البيانات التطبيقي",
    "هندسة ميكانيكية",
    "هندسة كهربائية"
  ],
  "search_queries_english": [
    "Applied Engineering",
    "Technical Solutions",
    "Product Development",
    "Engineering Design",
    "Applied Technology",
    "Software Eng

In [11]:
# STEP 6: Search in Database Based on Path Type

print("Searching database...")
print()

if path_type == "track_only":

    # Search in tracks only
    print("Tracks Results:")
    tracks = track_vs.similarity_search(track_analysis_response.content, k=3)
    for i, doc in enumerate(tracks, 1):
        print(f"\n[{i}] Metadata:", doc.metadata)
        print(doc.page_content[:500], "...")

else:

    # Search in colleges
    print("Colleges Results:")
    colleges = college_vs.similarity_search(college_analysis_response.content, k=3)
    for i, doc in enumerate(colleges, 1):
        print(f"\n[{i}] Metadata:", doc.metadata)
        print(doc.page_content[:500], "...")

    print("\n" + "="*60 + "\n")

    # Search in tracks
    print("Tracks Results:")
    tracks = track_vs.similarity_search(track_analysis_response.content, k=3)
    for i, doc in enumerate(tracks, 1):
        print(f"\n[{i}] Metadata:", doc.metadata)
        print(doc.page_content[:500], "...")

print("\n" + "="*60)
print("Search complete")
print("="*60)
print()


# STEP 7: Prepare Final Report Prompt Based on Path Type


print("="*60)
print(f"Preparing final report for {user_name}...")
print("="*60)
print()

# Prepare answers summary for final report
answers_summary = ""
for item in conversation_history:
    answers_summary += f"""
السؤال {item['question_number']}: {item['question']}
الإجابة: {item['answer']}
───────────────────────────────────
"""


if path_type == "track_only":

    # PATH 1: Tracks Only Report

    search_results_text = "نتائج البحث - التراكات التدريبية المتاحة:\n\n"

    for i, doc in enumerate(tracks[:10], 1):
        search_results_text += f"""
تراك رقم {i}
Metadata: {doc.metadata}

{doc.page_content[:700]}

"""

    final_prompt = f"""
أنت مستشار مسارات تدريبية وخبير توجيه مهني، دورك تحليل البيانات بعمق ثم تقديم توصيات واضحة ومختصرة مبنية على الأدلة.

بيانات المتقدم:
الاسم: {user_name}

إجابات المقابلة:
{answers_summary.strip()}

تحليل الاهتمامات والمهارات:
{track_analysis_response.content}

نتائج البحث في قاعدة بيانات التراكات:
{search_results_text}

مهمتك:

اقرأ جميع المعطيات السابقة أولاً قراءة تحليلية كاملة، ثم قدّم تقريراً منظماً وسهل الفهم للمستخدم، دون حشو أو تكرار.

────────────────────
1) التحليل الشخصي

قدّم تحليلاً موجزاً في فقرتين فقط يوضح:
- الاهتمامات والميول الأساسية
- المهارات الظاهرة من إجاباته
- أسلوب التعلم المتوقع
- التوجه المهني العام

اكتب التحليل بأسلوب استشاري واضح، يعتمد على ما ورد فعلياً في إجابات المستخدم.

────────────────────
2) أفضل 3 تراكات تدريبية مقترحة

الشروط:
- اختر 3 تراكات فقط من نتائج البحث
- استخدم الاسم الرسمي للتراك كما ورد في الـ Metadata
- رتّبهم من الأنسب إلى الأقل مناسبة

لكل تراك استخدم التنسيق التالي:

التراك الأول:
اسم التراك: [الاسم الرسمي من النتائج]

سبب الترشيح:
اشرح في سطرين فقط لماذا هذا التراك مناسب للمستخدم، مع ربط مباشر بإجاباته.
مثال:
- ذكر المستخدم [اقتباس مختصر]، وهذا يدل على اهتمامه بـ [مهارة/مجال] الذي يُعد أساسياً في هذا التراك.

مدة التدريب: [من النتائج]
المهارات الأساسية: [من النتائج]
الوظائف المحتملة: [من النتائج]
تحدٍ محتمل: اذكر تحدياً واحداً واقعياً قد يواجهه المستخدم في هذا التراك.

(كرر نفس الهيكل للتراك الثاني والثالث)

────────────────────
3) نصائح عملية مخصصة

قدّم 3 إلى 4 نصائح قصيرة ومباشرة، مثل:
- ما الذي يجب أن يركز عليه في البداية
- مهارة أساسية يجب تطويرها مبكراً
- خطأ شائع يُفضل تجنبه بناءً على حالته

────────────────────
4) موارد مقترحة

اقترح موارد محددة وواقعية، مثل:
- منصات تعليمية مناسبة
- نوع الكورسات التي يجب البحث عنها
- مجتمعات أو بيئات تعلم مفيدة

قواعد مهمة:
- لا تذكر نسب أو أرقام تقييم
- لا تستخدم عبارات عامة أو تسويقية
- اربط كل استنتاج بجزء واضح من إجابات المستخدم
- اجعل التقرير واضحاً، مختصراً، ومفيداً فعلياً للمستخدم
"""


else:

    # PATH 2: Colleges + Tracks Report

    colleges_results_text = "نتائج البحث - الكليات والأقسام الجامعية:\n\n"

    for i, doc in enumerate(colleges[:10], 1):
        colleges_results_text += f"""
كلية/قسم رقم {i}
Metadata: {doc.metadata}

{doc.page_content[:700]}

"""

    tracks_results_text = "نتائج البحث - التراكات التدريبية:\n\n"

    for i, doc in enumerate(tracks[:10], 1):
        tracks_results_text += f"""
تراك رقم {i}
Metadata: {doc.metadata}

{doc.page_content[:700]}

"""

    final_prompt = f"""
أنت مستشار أكاديمي ومهني خبير في توجيه خريجي الثانوية، مهمتك تحليل البيانات بعمق ثم تقديم توصيات عملية واضحة تساعد الطالب على اتخاذ قرار واعٍ.

بيانات الطالب:
الاسم: {user_name}
الحالة: خريج ثانوية

إجابات المقابلة:
{answers_summary.strip()}

تحليل الاهتمامات الأكاديمية:
{college_analysis_response.content}

تحليل الميول والمسارات التدريبية:
{track_analysis_response.content}

نتائج البحث في قاعدة بيانات الكليات:
{colleges_results_text}

نتائج البحث في قاعدة بيانات التراكات:
{tracks_results_text}

مهمتك:

اقرأ جميع المعطيات السابقة أولاً قراءة تحليلية كاملة، ثم قدّم تقريراً منظماً ومختصراً للمستخدم، دون مبالغة أو تكرار.

────────────────────
1) التحليل الأكاديمي والمهني

قدّم تحليلاً في فقرتين فقط يوضح:
- المواد والمجالات التي يميل إليها الطالب
- أسلوب التعلم الأنسب له
- نقاط القوة الظاهرة من إجاباته
- مدى ملاءمته للمسار الجامعي بشكل عام

يجب أن يكون التحليل مبنياً فقط على إجابات الطالب الفعلية.

────────────────────
2) أفضل 3 كليات / أقسام جامعية مقترحة

الشروط:
- اختر 3 ترشيحات فقط من قاعدة بيانات الكليات
- استخدم الاسم الكامل والدقيق: الجامعة + الكلية + القسم
- رتّبهم من الأنسب إلى الأقل مناسبة

لكل ترشيح استخدم التنسيق التالي:

الترشيح الأول:
[اسم الجامعة] – [اسم الكلية] – [اسم القسم]

سبب الترشيح:
اشرح في سطرين فقط لماذا هذا القسم مناسب للطالب، مع ربط مباشر بإجاباته.
مثال:
- أشار الطالب إلى اهتمامه بـ [اقتباس مختصر]، وهو ما يتوافق مع طبيعة هذا القسم التي تعتمد على [جانب أكاديمي محدد].

المواد الأساسية: [من النتائج]
المهارات المطلوبة: [من النتائج]
المسارات المهنية المحتملة: [من النتائج]
تحدٍ محتمل: تحدٍ واحد واقعي قد يواجهه الطالب في هذا القسم.
نصيحة خاصة: نصيحة عملية إذا قرر الالتحاق بهذا القسم.

(كرر نفس الهيكل للترشيح الثاني والثالث)

────────────────────
3) أفضل 3 تراكات تدريبية مكملة للدراسة الجامعية

الشروط:
- اختر 3 تراكات فقط من قاعدة بيانات التراكات
- هذه التراكات داعمة للدراسة الجامعية وليست بديلاً عنها
- استخدم الاسم الرسمي الدقيق لكل تراك

لكل تراك استخدم التنسيق التالي:

التراك الأول:
[اسم التراك الرسمي]

كيف يدعم الدراسة الجامعية؟
اشرح في سطرين فقط كيف يساعد هذا التراك الطالب أثناء أو بعد الدراسة الجامعية، مع ربط واضح بتخصصه المقترح.

مدة التدريب: [من النتائج]
المهارات المكتسبة: [من النتائج]
القيمة المضافة: ميزة واحدة واضحة يكتسبها الطالب مقارنة بزملائه.

(كرر للتراك الثاني والثالث)

────────────────────
4) نصائح عملية مخصصة

قدّم 3 إلى 4 نقاط قصيرة تشمل:
- ما الذي يجب أن يبدأ به مبكراً
- مهارة أساسية يجب تطويرها من الآن
- خطأ شائع يُفضل تجنبه
- نصيحة شخصية مبنية على ملفه بالكامل

────────────────────
5) موارد مقترحة

اقترح موارد واضحة ومناسبة، مثل:
- مصادر للتحضير للدراسة الجامعية
- منصات مناسبة للتراكات التدريبية
- مجتمعات أو أدوات تدعم التطور المهني المبكر

قواعد أساسية:
- لا تستخدم نسب أو تقييمات رقمية
- لا تكتب جُملاً عامة أو إنشائية
- اربط كل توصية بإجابات الطالب
- وضّح الفرق بين المسار الأكاديمي والتدريب التطبيقي
- اجعل التقرير واضحاً، مختصراً، وسهل القراءة
"""



# STEP 8: Generate Final Report

messages = [
    SystemMessage(content=f"أنت مستشار {'تدريبي' if path_type == 'track_only' else 'أكاديمي ومهني'} خبير متخصص في تقديم توصيات دقيقة ومخصصة ومبنية على بيانات حقيقية."),
    HumanMessage(content=final_prompt)
]

print("Generating comprehensive report to help you map your academic and career path...")
print()

final_report = llm.invoke(messages)

print()
print(final_report.content)
print()
print("Report complete")


Searching database...

Colleges Results:

[1] Metadata: {'type': 'faculty_department_profile', 'id': 'alexcommerce_mathematics'}
الجامعة: جامعة الإسكندرية
    الكلية: كلية التجارة
    القسم: قسم الرياضيات

    وصف البرنامج:
    قسم الرياضيات يدرس الرياضيات البحتة والتطبيقية، الاحتمالات، والنمذجة الرياضية للتطبيقات العملية المختلفة.

    الاهتمامات:
    الرياضيات البحتة, الرياضيات التطبيقية, الاحتمالات

    المواد الأساسية:
    الرياضيات, المنطق, الإحصاء

    المهارات المطلوبة:
    حل المشكلات, التفكير التحليلي, النمذجة

    أسلوب الدراسة:
    نظري, بحثي, عملي

    المسارات المهنية:
    رياضي, محلل مالي, خبير اكتواري

     ...

[2] Metadata: {'type': 'faculty_department_profile', 'id': 'ainshams_science_physics_computer_science'}
المسارات المهنية:
    عالم حاسوبي, عالم بيانات, مهندس برمجيات (تقنية عالية), مطور خوارزميات

    --- English Keywords ---
    Department: Physics / Computer Science
    Interests: computational physics, scientific computing, algorithm design, modeling and simul